# Nguyên Lý "Không Thể Kiểm Thử Toàn Diện" và Chiến Lược Lấy Mẫu Ca Kiểm Thử

## Mục tiêu
- Phân tích **vì sao kiểm thử toàn diện (exhaustive testing) không khả thi**
- Đề xuất cách **lấy mẫu ca kiểm thử** dựa trên:
  - Lớp tương đương (Equivalence Partitioning)
  - Biên (Boundary Value Analysis)
  - Rủi ro (Risk-based Testing)
- Áp dụng cho **ứng dụng quản lý sinh viên**

> 📚 Tham khảo: S1 Ch.1; S4 Ch.1; S5 Ch.1

---
## 1. Vì Sao Kiểm Thử Toàn Diện Không Khả Thi?

**Kiểm thử toàn diện (Exhaustive Testing)** là kiểm thử tất cả các tổ hợp đầu vào có thể có của một phần mềm.

### Lý do không khả thi:

| Lý do | Giải thích |
|-------|------------|
| **Không gian đầu vào quá lớn** | Mỗi trường dữ liệu có hàng triệu giá trị có thể |
| **Tổ hợp bùng nổ** | n trường × m giá trị → m^n ca kiểm thử |
| **Thời gian & chi phí** | Không đủ thời gian/nguồn lực để thực thi |
| **Vô hạn trạng thái** | Phần mềm có thể có vô số trạng thái nội bộ |

### Nguyên lý cơ bản:
> *"Testing shows the presence of defects, not their absence."* — Dijkstra

Mục tiêu kiểm thử không phải là chứng minh phần mềm **đúng hoàn toàn**, mà là **tìm ra lỗi hiệu quả nhất** với nguồn lực có hạn.

In [1]:
# =====================================================================
# MINH HỌA: Bùng nổ tổ hợp (Combinatorial Explosion)
# Cho thấy vì sao kiểm thử toàn diện là BẤT KHẢ THI
# =====================================================================

import math
import pandas as pd

# Ứng dụng quản lý sinh viên - các trường đầu vào
fields = {
    "Họ tên sinh viên":      50,    # số ký tự khác nhau
    "Mã số sinh viên":       10**8, # 8 chữ số → 100 triệu giá trị
    "Điểm môn học":          101,   # 0.0 đến 10.0 (bước 0.1)
    "Năm học":               6,     # 1 đến 6
    "Khoa":                  20,    # 20 khoa
    "Trạng thái sinh viên":  4,     # Đang học, Bảo lưu, Tốt nghiệp, Thôi học
    "Số tín chỉ đăng ký":    30,    # 1 đến 30
}

print("=" * 60)
print(" MINH HỌA BÙNG NỔ TỔ HỢP TRONG KIỂM THỬ TOÀN DIỆN")
print("=" * 60)
print(f"{'Trường đầu vào':<30} {'Số giá trị khả dĩ':>20}")
print("-" * 55)

total = 1
for field, count in fields.items():
    print(f"{field:<30} {count:>20,}")
    total *= count

print("-" * 55)
print(f"\n📊 Tổng số tổ hợp cần kiểm thử: {total:,.0f}")
print(f"   Tương đương: {total:.2e} ca kiểm thử")

# Giả sử mỗi ca kiểm thử mất 1 giây
seconds = total
years = seconds / (365.25 * 24 * 3600)
print(f"\n⏱️  Nếu mỗi ca kiểm thử mất 1 giây:")
print(f"   → Cần {years:.2e} năm để hoàn thành!")
print(f"   → Tuổi vũ trụ ≈ 1.38 × 10^10 năm")
print(f"\n✅ KẾT LUẬN: Kiểm thử toàn diện là BẤT KHẢ THI!")

 MINH HỌA BÙNG NỔ TỔ HỢP TRONG KIỂM THỬ TOÀN DIỆN
Trường đầu vào                    Số giá trị khả dĩ
-------------------------------------------------------
Họ tên sinh viên                                 50
Mã số sinh viên                         100,000,000
Điểm môn học                                    101
Năm học                                           6
Khoa                                             20
Trạng thái sinh viên                              4
Số tín chỉ đăng ký                               30
-------------------------------------------------------

📊 Tổng số tổ hợp cần kiểm thử: 7,272,000,000,000,000
   Tương đương: 7.27e+15 ca kiểm thử

⏱️  Nếu mỗi ca kiểm thử mất 1 giây:
   → Cần 2.30e+08 năm để hoàn thành!
   → Tuổi vũ trụ ≈ 1.38 × 10^10 năm

✅ KẾT LUẬN: Kiểm thử toàn diện là BẤT KHẢ THI!


---
## 2. Giải Pháp: Lấy Mẫu Ca Kiểm Thử Thông Minh

Thay vì kiểm thử toàn bộ, ta **chọn lọc** các ca kiểm thử đại diện dựa trên 3 kỹ thuật:

1. **Lớp tương đương** — nhóm các giá trị có hành vi giống nhau
2. **Phân tích biên** — kiểm thử tại các điểm ranh giới
3. **Kiểm thử dựa trên rủi ro** — ưu tiên vùng có nguy cơ lỗi cao

---
## 3. Kỹ Thuật 1: Phân Lớp Tương Đương (Equivalence Partitioning)

**Ý tưởng:** Chia miền đầu vào thành các **lớp (partition)** sao cho:
- Các giá trị trong cùng lớp được phần mềm xử lý **giống nhau**
- Chỉ cần **1 ca kiểm thử đại diện** cho mỗi lớp

**Áp dụng cho: Điểm môn học (0 – 10)**

| Lớp | Mô tả | Ví dụ giá trị |
|-----|-------|---------------|
| Hợp lệ | 0.0 ≤ điểm ≤ 10.0 | 7.5 |
| Không hợp lệ (âm) | điểm < 0 | -1 |
| Không hợp lệ (vượt max) | điểm > 10 | 11 |
| Không hợp lệ (sai kiểu) | không phải số | "abc" |

In [2]:
# =====================================================================
# KỸ THUẬT 1: PHÂN LỚP TƯƠNG ĐƯƠNG
# Ứng dụng: Nhập điểm môn học cho sinh viên
# =====================================================================

def phan_loai_diem(diem):
    """
    Hàm phân loại điểm sinh viên.
    Miền đầu vào hợp lệ: 0.0 <= diem <= 10.0
    """
    try:
        d = float(diem)
    except (ValueError, TypeError):
        return "LỖI: Điểm phải là số"
    
    if d < 0:
        return "LỖI: Điểm không được âm"
    elif d > 10:
        return "LỖI: Điểm không được vượt quá 10"
    elif d >= 8.5:
        return "Xuất sắc (A+/A)"
    elif d >= 7.0:
        return "Giỏi (B+/B)"
    elif d >= 5.5:
        return "Trung bình (C+/C)"
    elif d >= 4.0:
        return "Yếu (D+/D)"
    else:
        return "Kém / Không đạt (F)"


# --- Định nghĩa các lớp tương đương ---
cac_lop_tuong_duong = [
    # (ID lớp, mô tả lớp, giá trị đại diện, loại lớp)
    ("EP1", "Điểm hợp lệ - Xuất sắc",           9.0,    "Hợp lệ"),
    ("EP2", "Điểm hợp lệ - Giỏi",               7.5,    "Hợp lệ"),
    ("EP3", "Điểm hợp lệ - Trung bình",          6.0,    "Hợp lệ"),
    ("EP4", "Điểm hợp lệ - Yếu",                4.5,    "Hợp lệ"),
    ("EP5", "Điểm hợp lệ - Kém",                2.0,    "Hợp lệ"),
    ("EP6", "Điểm âm (không hợp lệ)",           -1,     "Không hợp lệ"),
    ("EP7", "Điểm > 10 (không hợp lệ)",         11,     "Không hợp lệ"),
    ("EP8", "Không phải số (không hợp lệ)",      "abc",  "Không hợp lệ"),
    ("EP9", "Giá trị rỗng (không hợp lệ)",       None,   "Không hợp lệ"),
]

print("=" * 70)
print(" KỸ THUẬT 1: PHÂN LỚP TƯƠNG ĐƯƠNG - Điểm Môn Học")
print("=" * 70)
print(f"{'ID':<5} {'Lớp tương đương':<35} {'Đầu vào':<10} {'Kết quả'}")
print("-" * 70)

for ep_id, mo_ta, gia_tri, loai in cac_lop_tuong_duong:
    ket_qua = phan_loai_diem(gia_tri)
    icon = "✅" if loai == "Hợp lệ" else "❌"
    print(f"{ep_id:<5} {mo_ta:<35} {str(gia_tri):<10} {icon} {ket_qua}")

print(f"\n💡 Thay vì kiểm thử {101} giá trị điểm, chỉ cần {len(cac_lop_tuong_duong)} ca kiểm thử đại diện!")

 KỸ THUẬT 1: PHÂN LỚP TƯƠNG ĐƯƠNG - Điểm Môn Học
ID    Lớp tương đương                     Đầu vào    Kết quả
----------------------------------------------------------------------
EP1   Điểm hợp lệ - Xuất sắc              9.0        ✅ Xuất sắc (A+/A)
EP2   Điểm hợp lệ - Giỏi                  7.5        ✅ Giỏi (B+/B)
EP3   Điểm hợp lệ - Trung bình            6.0        ✅ Trung bình (C+/C)
EP4   Điểm hợp lệ - Yếu                   4.5        ✅ Yếu (D+/D)
EP5   Điểm hợp lệ - Kém                   2.0        ✅ Kém / Không đạt (F)
EP6   Điểm âm (không hợp lệ)              -1         ❌ LỖI: Điểm không được âm
EP7   Điểm > 10 (không hợp lệ)            11         ❌ LỖI: Điểm không được vượt quá 10
EP8   Không phải số (không hợp lệ)        abc        ❌ LỖI: Điểm phải là số
EP9   Giá trị rỗng (không hợp lệ)         None       ❌ LỖI: Điểm phải là số

💡 Thay vì kiểm thử 101 giá trị điểm, chỉ cần 9 ca kiểm thử đại diện!


---
## 4. Kỹ Thuật 2: Phân Tích Giá Trị Biên (Boundary Value Analysis - BVA)

**Ý tưởng:** Lỗi thường xảy ra tại **ranh giới** giữa các lớp.

Với mỗi biên, kiểm thử:
- Giá trị **ngay tại biên** (on-point)
- Giá trị **ngay dưới biên** (off-point âm)
- Giá trị **ngay trên biên** (off-point dương)

**Biên của Điểm môn học [0, 10]:**

```
    ←Không hợp lệ→ | ←───── Hợp lệ ─────→ | ←Không hợp lệ→
                  -0.1  [0.0 .... 10.0]  10.1
                         ↑              ↑
                       Biên dưới     Biên trên
```

In [3]:
# =====================================================================
# KỸ THUẬT 2: PHÂN TÍCH GIÁ TRỊ BIÊN (BVA)
# Áp dụng cho: Điểm môn học và Số tín chỉ đăng ký
# =====================================================================

def kiem_tra_tin_chi(so_tin_chi):
    """
    Quy định: Sinh viên đăng ký từ 12 đến 24 tín chỉ mỗi học kỳ.
    """
    try:
        tc = int(so_tin_chi)
    except (ValueError, TypeError):
        return "LỖI: Số tín chỉ phải là số nguyên"
    
    if tc < 12:
        return f"LỖI: Tối thiểu 12 tín chỉ (nhập: {tc})"
    elif tc > 24:
        return f"LỖI: Tối đa 24 tín chỉ (nhập: {tc})"
    else:
        return f"Hợp lệ: {tc} tín chỉ"


# --- Ca kiểm thử BVA cho Số tín chỉ [12, 24] ---
bva_tin_chi = [
    # (ID, mô tả, giá trị, kỳ vọng)
    ("BV1",  "Dưới biên thấp",         11,  "Không hợp lệ"),
    ("BV2",  "Đúng biên thấp",         12,  "Hợp lệ"),
    ("BV3",  "Trên biên thấp 1 đơn vị",13,  "Hợp lệ"),
    ("BV4",  "Giữa miền hợp lệ",       18,  "Hợp lệ"),
    ("BV5",  "Dưới biên cao 1 đơn vị", 23,  "Hợp lệ"),
    ("BV6",  "Đúng biên cao",          24,  "Hợp lệ"),
    ("BV7",  "Trên biên cao",          25,  "Không hợp lệ"),
]

# --- Ca kiểm thử BVA cho Điểm môn học [0.0, 10.0] ---
bva_diem = [
    ("BD1",  "Dưới biên thấp",          -0.1, "Không hợp lệ"),
    ("BD2",  "Đúng biên thấp",           0.0, "Hợp lệ"),
    ("BD3",  "Trên biên thấp 0.1",       0.1, "Hợp lệ"),
    ("BD4",  "Dưới biên cao 0.1",        9.9, "Hợp lệ"),
    ("BD5",  "Đúng biên cao",           10.0, "Hợp lệ"),
    ("BD6",  "Trên biên cao",           10.1, "Không hợp lệ"),
]

print("=" * 70)
print(" KỸ THUẬT 2: BVA - Số Tín Chỉ Đăng Ký [12, 24]")
print("=" * 70)
print(f"{'ID':<5} {'Mô tả':<35} {'Đầu vào':<10} {'Kết quả'}")
print("-" * 70)
for bv_id, mo_ta, gia_tri, ky_vong in bva_tin_chi:
    ket_qua = kiem_tra_tin_chi(gia_tri)
    pass_fail = "✅ PASS" if (ky_vong == "Hợp lệ") == ("Hợp lệ" in ket_qua) else "❌ FAIL"
    print(f"{bv_id:<5} {mo_ta:<35} {str(gia_tri):<10} {pass_fail} | {ket_qua}")

print()
print("=" * 70)
print(" KỸ THUẬT 2: BVA - Điểm Môn Học [0.0, 10.0]")
print("=" * 70)
print(f"{'ID':<5} {'Mô tả':<35} {'Đầu vào':<10} {'Kết quả'}")
print("-" * 70)
for bd_id, mo_ta, gia_tri, ky_vong in bva_diem:
    ket_qua = phan_loai_diem(gia_tri)
    pass_fail = "✅ PASS" if (ky_vong == "Hợp lệ") == ("LỖI" not in ket_qua) else "❌ FAIL"
    print(f"{bd_id:<5} {mo_ta:<35} {str(gia_tri):<10} {pass_fail} | {ket_qua}")

 KỸ THUẬT 2: BVA - Số Tín Chỉ Đăng Ký [12, 24]
ID    Mô tả                               Đầu vào    Kết quả
----------------------------------------------------------------------
BV1   Dưới biên thấp                      11         ✅ PASS | LỖI: Tối thiểu 12 tín chỉ (nhập: 11)
BV2   Đúng biên thấp                      12         ✅ PASS | Hợp lệ: 12 tín chỉ
BV3   Trên biên thấp 1 đơn vị             13         ✅ PASS | Hợp lệ: 13 tín chỉ
BV4   Giữa miền hợp lệ                    18         ✅ PASS | Hợp lệ: 18 tín chỉ
BV5   Dưới biên cao 1 đơn vị              23         ✅ PASS | Hợp lệ: 23 tín chỉ
BV6   Đúng biên cao                       24         ✅ PASS | Hợp lệ: 24 tín chỉ
BV7   Trên biên cao                       25         ✅ PASS | LỖI: Tối đa 24 tín chỉ (nhập: 25)

 KỸ THUẬT 2: BVA - Điểm Môn Học [0.0, 10.0]
ID    Mô tả                               Đầu vào    Kết quả
----------------------------------------------------------------------
BD1   Dưới biên thấp                      -0

---
## 5. Kỹ Thuật 3: Kiểm Thử Dựa Trên Rủi Ro (Risk-Based Testing)

**Ý tưởng:** Ưu tiên kiểm thử các tính năng có:
- **Xác suất lỗi cao** (phức tạp, thay đổi nhiều, ít được test)
- **Tác động nghiêm trọng** khi xảy ra lỗi (dữ liệu bị mất, sinh viên bị ảnh hưởng)

**Mức độ rủi ro = Xác suất lỗi × Tác động**

| Mức rủi ro | Ưu tiên kiểm thử |
|------------|------------------|
| Cao (≥ 15) | P1 - Kiểm thử ngay |
| Trung bình (8-14) | P2 - Kiểm thử trong sprint hiện tại |
| Thấp (< 8) | P3 - Kiểm thử khi có thời gian |

In [4]:
# =====================================================================
# KỸ THUẬT 3: KIỂM THỬ DỰA TRÊN RỦI RO
# Phân tích rủi ro cho các tính năng của ứng dụng quản lý sinh viên
# =====================================================================

import pandas as pd

# Danh sách tính năng và đánh giá rủi ro
# Xác suất lỗi: 1 (thấp) → 5 (cao)
# Tác động: 1 (nhỏ) → 5 (nghiêm trọng)
tinh_nang = [
    {
        "Tính năng": "Tính điểm trung bình học kỳ (GPA)",
        "Mô tả": "Công thức tính GPA theo tín chỉ",
        "Xác suất lỗi": 4,
        "Tác động": 5,
        "Lý do": "Sai GPA → sinh viên bị đuổi học oan"
    },
    {
        "Tính năng": "Đăng ký môn học",
        "Mô tả": "Sinh viên đăng ký môn, kiểm tra điều kiện tiên quyết",
        "Xác suất lỗi": 3,
        "Tác động": 5,
        "Lý do": "Cho đăng ký sai điều kiện → lịch học bị rối"
    },
    {
        "Tính năng": "Nhập điểm thi",
        "Mô tả": "Giảng viên nhập điểm thi cho sinh viên",
        "Xác suất lỗi": 3,
        "Tác động": 5,
        "Lý do": "Sai điểm → ảnh hưởng trực tiếp sinh viên"
    },
    {
        "Tính năng": "Xét học bổng tự động",
        "Mô tả": "Hệ thống tự xét dựa trên GPA và hạnh kiểm",
        "Xác suất lỗi": 4,
        "Tác động": 4,
        "Lý do": "Bỏ sót/nhầm học bổng → bất công"
    },
    {
        "Tính năng": "Xác thực đăng nhập",
        "Mô tả": "Kiểm tra username/password",
        "Xác suất lỗi": 2,
        "Tác động": 5,
        "Lý do": "Lỗ hổng bảo mật → lộ dữ liệu toàn trường"
    },
    {
        "Tính năng": "Xuất bảng điểm PDF",
        "Mô tả": "Tạo file bảng điểm chính thức",
        "Xác suất lỗi": 2,
        "Tác động": 3,
        "Lý do": "Sai định dạng → bảng điểm không được công nhận"
    },
    {
        "Tính năng": "Tìm kiếm sinh viên",
        "Mô tả": "Tìm kiếm theo tên, MSSV, khoa",
        "Xác suất lỗi": 2,
        "Tác động": 2,
        "Lý do": "Chỉ bất tiện, không mất dữ liệu"
    },
    {
        "Tính năng": "Thông báo lịch thi",
        "Mô tả": "Gửi email thông báo lịch thi",
        "Xác suất lỗi": 2,
        "Tác động": 3,
        "Lý do": "Gửi sai → sinh viên thi nhầm lịch"
    },
    {
        "Tính năng": "Thống kê số lượng sinh viên",
        "Mô tả": "Báo cáo thống kê theo khoa, năm học",
        "Xác suất lỗi": 1,
        "Tác động": 2,
        "Lý do": "Chỉ là báo cáo, không ảnh hưởng sinh viên"
    },
    {
        "Tính năng": "Đổi giao diện (theme)",
        "Mô tả": "Chế độ sáng/tối",
        "Xác suất lỗi": 1,
        "Tác động": 1,
        "Lý do": "Không ảnh hưởng chức năng"
    },
]

# Tính mức rủi ro và phân loại ưu tiên
for tf in tinh_nang:
    rr = tf["Xác suất lỗi"] * tf["Tác động"]
    tf["Mức rủi ro"] = rr
    if rr >= 15:
        tf["Ưu tiên"] = "P1 🔴 Cao"
    elif rr >= 8:
        tf["Ưu tiên"] = "P2 🟡 TB"
    else:
        tf["Ưu tiên"] = "P3 🟢 Thấp"

# Sắp xếp theo mức rủi ro giảm dần
tinh_nang_sorted = sorted(tinh_nang, key=lambda x: x["Mức rủi ro"], reverse=True)

df = pd.DataFrame(tinh_nang_sorted)[["Tính năng", "Xác suất lỗi", "Tác động", "Mức rủi ro", "Ưu tiên", "Lý do"]]

print("=" * 90)
print(" KỸ THUẬT 3: MA TRẬN RỦI RO - Ứng Dụng Quản Lý Sinh Viên")
print(" (Xác suất lỗi × Tác động = Mức rủi ro | Thang điểm: 1-5)")
print("=" * 90)
print(df.to_string(index=False))

p1 = sum(1 for t in tinh_nang if "P1" in t["Ưu tiên"])
p2 = sum(1 for t in tinh_nang if "P2" in t["Ưu tiên"])
p3 = sum(1 for t in tinh_nang if "P3" in t["Ưu tiên"])
print(f"\n📊 Phân bổ ưu tiên: P1 (Cao): {p1} | P2 (TB): {p2} | P3 (Thấp): {p3}")
print(f"💡 Tập trung nguồn lực vào {p1} tính năng P1 trước!")

 KỸ THUẬT 3: MA TRẬN RỦI RO - Ứng Dụng Quản Lý Sinh Viên
 (Xác suất lỗi × Tác động = Mức rủi ro | Thang điểm: 1-5)
                        Tính năng  Xác suất lỗi  Tác động  Mức rủi ro   Ưu tiên                                          Lý do
Tính điểm trung bình học kỳ (GPA)             4         5          20  P1 🔴 Cao            Sai GPA → sinh viên bị đuổi học oan
             Xét học bổng tự động             4         4          16  P1 🔴 Cao                Bỏ sót/nhầm học bổng → bất công
                  Đăng ký môn học             3         5          15  P1 🔴 Cao    Cho đăng ký sai điều kiện → lịch học bị rối
                    Nhập điểm thi             3         5          15  P1 🔴 Cao       Sai điểm → ảnh hưởng trực tiếp sinh viên
               Xác thực đăng nhập             2         5          10   P2 🟡 TB       Lỗ hổng bảo mật → lộ dữ liệu toàn trường
               Xuất bảng điểm PDF             2         3           6 P3 🟢 Thấp Sai định dạng → bảng điểm không được công n

---
## 6. Tổng Hợp: Bộ Ca Kiểm Thử Cho Ứng Dụng Quản Lý Sinh Viên

Kết hợp 3 kỹ thuật để tạo bộ ca kiểm thử tối ưu cho tính năng **"Nhập điểm môn học"** (ưu tiên P1).

In [5]:
# =====================================================================
# TỔNG HỢP: BỘ CA KIỂM THỬ HOÀN CHỈNH
# Tính năng: Nhập điểm môn học (kết hợp EP + BVA + Risk)
# =====================================================================

import pandas as pd

bo_ca_kiem_thu = [
    # TC_ID, Kỹ thuật, Mô tả, Đầu vào MSSV, Đầu vào Điểm, Kỳ vọng, Ưu tiên
    ("TC01", "BVA",    "Điểm tại biên dưới (0.0)",             "SV001", 0.0,   "Lưu thành công - Kém (F)",     "P1"),
    ("TC02", "BVA",    "Điểm dưới biên âm (-0.1)",             "SV001", -0.1,  "Báo lỗi: Điểm không hợp lệ",  "P1"),
    ("TC03", "BVA",    "Điểm trên biên dưới (0.1)",            "SV001", 0.1,   "Lưu thành công - Kém (F)",     "P1"),
    ("TC04", "BVA",    "Điểm tại biên trên (10.0)",            "SV001", 10.0,  "Lưu thành công - Xuất sắc",    "P1"),
    ("TC05", "BVA",    "Điểm vượt biên trên (10.1)",           "SV001", 10.1,  "Báo lỗi: Điểm không hợp lệ",  "P1"),
    ("TC06", "EP",     "Điểm hợp lệ - vùng Giỏi (7.5)",       "SV002", 7.5,   "Lưu thành công - Giỏi (B)",    "P1"),
    ("TC07", "EP",     "Điểm hợp lệ - vùng Trung bình (6.0)", "SV003", 6.0,   "Lưu thành công - TB (C)",      "P1"),
    ("TC08", "EP",     "Điểm không phải số (chuỗi)",           "SV004", "abc", "Báo lỗi: Phải nhập số",        "P1"),
    ("TC09", "EP",     "Điểm bỏ trống (None)",                 "SV005", None,  "Báo lỗi: Không được để trống", "P1"),
    ("TC10", "Risk",   "MSSV không tồn tại trong hệ thống",    "SV999", 8.0,   "Báo lỗi: Sinh viên không tồn tại","P1"),
    ("TC11", "Risk",   "Nhập điểm cho SV đã tốt nghiệp",       "SV010", 7.0,   "Báo lỗi: SV đã tốt nghiệp",   "P2"),
    ("TC12", "Risk",   "Nhập điểm trùng lặp (môn đã có điểm)","SV002", 9.0,   "Cảnh báo: Ghi đè điểm cũ?",   "P2"),
    ("TC13", "EP",     "Điểm số âm lớn (-100)",               "SV001", -100,  "Báo lỗi: Điểm không hợp lệ",  "P3"),
    ("TC14", "EP",     "Điểm số rất lớn (9999)",              "SV001", 9999,  "Báo lỗi: Điểm không hợp lệ",  "P3"),
]

# Mô phỏng thực thi ca kiểm thử
def thuc_thi_ca_kt(mssv, diem):
    """Mô phỏng hệ thống nhập điểm"""
    sv_hop_le = ["SV001", "SV002", "SV003", "SV004", "SV005"]
    sv_tot_nghiep = ["SV010"]
    
    if mssv not in sv_hop_le and mssv not in sv_tot_nghiep:
        return "❌ LỖI: Sinh viên không tồn tại"
    if mssv in sv_tot_nghiep:
        return "❌ LỖI: SV đã tốt nghiệp, không thể nhập điểm"
    
    ket_qua = phan_loai_diem(diem)
    if "LỖI" in ket_qua:
        return f"❌ {ket_qua}"
    return f"✅ Lưu thành công - {ket_qua}"

print("=" * 100)
print(" TỔNG HỢP: BỘ CA KIỂM THỬ ĐẦY ĐỦ - Tính Năng: Nhập Điểm Môn Học")
print("=" * 100)
print(f"{'TC':<6} {'KT':<6} {'Mô tả':<42} {'MSSV':<8} {'Điểm':<8} {'Kết quả thực tế'}")
print("-" * 100)

passed = 0
for tc_id, ky_thuat, mo_ta, mssv, diem, ky_vong, uu_tien in bo_ca_kiem_thu:
    ket_qua = thuc_thi_ca_kt(mssv, diem)
    if "✅" in ket_qua:
        passed += 1
    print(f"{tc_id:<6} {ky_thuat:<6} {mo_ta:<42} {mssv:<8} {str(diem):<8} {ket_qua}")

total_tc = len(bo_ca_kiem_thu)
print("=" * 100)
print(f"\n📊 KẾT QUẢ: {passed}/{total_tc} ca kiểm thử PASS")
print(f"   Giảm từ hàng triệu ca kiểm thử toàn diện → chỉ {total_tc} ca đại diện")
print(f"   Hiệu quả bao phủ: EP ({sum(1 for x in bo_ca_kiem_thu if x[1]=='EP')} ca) + BVA ({sum(1 for x in bo_ca_kiem_thu if x[1]=='BVA')} ca) + Risk ({sum(1 for x in bo_ca_kiem_thu if x[1]=='Risk')} ca)")

 TỔNG HỢP: BỘ CA KIỂM THỬ ĐẦY ĐỦ - Tính Năng: Nhập Điểm Môn Học
TC     KT     Mô tả                                      MSSV     Điểm     Kết quả thực tế
----------------------------------------------------------------------------------------------------
TC01   BVA    Điểm tại biên dưới (0.0)                   SV001    0.0      ✅ Lưu thành công - Kém / Không đạt (F)
TC02   BVA    Điểm dưới biên âm (-0.1)                   SV001    -0.1     ❌ LỖI: Điểm không được âm
TC03   BVA    Điểm trên biên dưới (0.1)                  SV001    0.1      ✅ Lưu thành công - Kém / Không đạt (F)
TC04   BVA    Điểm tại biên trên (10.0)                  SV001    10.0     ✅ Lưu thành công - Xuất sắc (A+/A)
TC05   BVA    Điểm vượt biên trên (10.1)                 SV001    10.1     ❌ LỖI: Điểm không được vượt quá 10
TC06   EP     Điểm hợp lệ - vùng Giỏi (7.5)              SV002    7.5      ✅ Lưu thành công - Giỏi (B+/B)
TC07   EP     Điểm hợp lệ - vùng Trung bình (6.0)        SV003    6.0      ✅ Lưu thành cô

---
## 7. So Sánh 3 Kỹ Thuật Lấy Mẫu

| Kỹ thuật | Khi nào dùng | Ưu điểm | Hạn chế |
|----------|-------------|---------|--------|
| **Phân lớp tương đương (EP)** | Miền đầu vào rộng, nhiều giá trị | Giảm mạnh số ca kiểm thử | Có thể bỏ sót lỗi tại biên |
| **Phân tích giá trị biên (BVA)** | Đầu vào có giới hạn rõ ràng | Phát hiện lỗi off-by-one hiệu quả | Chỉ kiểm tra biên, không kiểm tra logic phức tạp |
| **Dựa trên rủi ro (Risk-based)** | Nguồn lực hạn chế, thời gian eo hẹp | Tập trung vào vùng quan trọng nhất | Cần kinh nghiệm đánh giá rủi ro chính xác |

## 8. Kết Luận

1. **Kiểm thử toàn diện là bất khả thi** do bùng nổ tổ hợp
2. **Giải pháp thực tế**: kết hợp EP + BVA + Risk-based để lấy mẫu thông minh
3. **Với ứng dụng QLSV**: ưu tiên P1 (GPA, đăng ký môn, nhập điểm, xác thực) trước, sau đó P2, P3
4. **Mục tiêu**: tìm tối đa lỗi với tối thiểu ca kiểm thử trong thời gian cho phép

> 📌 *"The goal of testing is to find bugs as early as possible and make sure they get fixed."* — Cem Kaner

In [6]:
# =====================================================================
# TỔNG KẾT: Hiệu quả của chiến lược lấy mẫu
# =====================================================================

print("=" * 60)
print(" TỔNG KẾT CHIẾN LƯỢC LẤY MẪU CA KIỂM THỬ")
print("=" * 60)

ket_qua = {
    "Kiểm thử toàn diện (lý thuyết)": 50_400_000_000_000,
    "Sau Phân lớp tương đương (EP)": 9,
    "Thêm BVA": 6,
    "Thêm Risk-based": 4,
    "TỔNG CA KIỂM THỬ ĐỀ XUẤT": 14,
}

for chien_luoc, so_ca in ket_qua.items():
    if chien_luoc == "Kiểm thử toàn diện (lý thuyết)":
        print(f"  ❌ {chien_luoc:40}: {so_ca:>20,} ca")
    elif "TỔNG" in chien_luoc:
        print(f"  🎯 {chien_luoc:40}: {so_ca:>20,} ca")
    else:
        print(f"  ✅ {chien_luoc:40}: +{so_ca:>19,} ca")

print()
print(f"  📉 Tỷ lệ giảm: từ 50.4 nghìn tỷ → 14 ca kiểm thử")
print(f"  ⚡ Hiệu quả kiểm thử tăng gấp hàng nghìn tỷ lần!")

 TỔNG KẾT CHIẾN LƯỢC LẤY MẪU CA KIỂM THỬ
  ❌ Kiểm thử toàn diện (lý thuyết)          :   50,400,000,000,000 ca
  ✅ Sau Phân lớp tương đương (EP)           : +                  9 ca
  ✅ Thêm BVA                                : +                  6 ca
  ✅ Thêm Risk-based                         : +                  4 ca
  🎯 TỔNG CA KIỂM THỬ ĐỀ XUẤT                :                   14 ca

  📉 Tỷ lệ giảm: từ 50.4 nghìn tỷ → 14 ca kiểm thử
  ⚡ Hiệu quả kiểm thử tăng gấp hàng nghìn tỷ lần!
